# Лабораторная работа №18: Быстрые samplers и distillation компактной diffusion model

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Переиспользовать checkpoint lab 17.
- Сравнить DDPM, DDIM и дополнительный solver.
- Обучить compact student и построить Pareto frontier.

## Теоретическая часть

DDIM использует тот же denoiser, но может идти по сокращённой последовательности timesteps детерминированно. Distillation обучает student воспроизводить несколько teacher steps одним шагом. Сравнение строится при одинаковых initial noise и labels.


DDIM вычисляет x̂0 и делает детерминированный переход между точками ᾱ. DPM-Solver++ использует тот же обученный ε-предсказатель. Progressive distillation аппроксимирует двумя teacher-переходами один student-переход; teacher заморожен.

Детерминированный DDIM-переход: $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\hat\epsilon)/\sqrt{\bar\alpha_t}$, $x_s=\sqrt{\bar\alpha_s}\hat x_0+\sqrt{1-\bar\alpha_s}\hat\epsilon$. При конечном $s=-1$ полагаем $\bar\alpha_{-1}=1$.

### Экспериментальный протокол

Обязательная зависимость — реально обученные веса ddpm.pt, classifier.pt и classifier_validation.json из lab 17. Используйте те же классы, noise и guidance=3 при сравнении. Smoke не является подтверждением качества distillation. Для длинных запусков обеспечьте постоянный каталог artifacts (Drive).

### Материалы

- [DDIM](https://arxiv.org/abs/2010.02502)
- [Progressive Distillation](https://arxiv.org/abs/2202.00512)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.

Версии библиотек, источник данных и лицензия должны быть указаны в отчёте.

## Реализация

`FAST_DEV_RUN=True` проверяет связность пайплайна на малом бюджете и не является итоговым экспериментом. Полная сдача требует `FAST_DEV_RUN=False` и реально полученных метрик.

In [1]:
from pathlib import Path
import os, json, random, time
import numpy as np
import torch
SEED = 42
FAST_DEV_RUN = False  # True только для короткой проверки связности, не для сдачи.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
# локально работаю на Apple Silicon, поэтому кроме CUDA проверяю MPS
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
ARTIFACTS = Path(os.environ.get('CV_ARTIFACTS', 'artifacts'))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def save_json(name, value):
    destination = ARTIFACTS / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, ensure_ascii=False, indent=2))
print({'device':DEVICE,'fast_dev_run':FAST_DEV_RUN,'artifacts':str(ARTIFACTS)})

{'device': 'mps', 'fast_dev_run': False, 'artifacts': 'artifacts'}


### Проверка обученного teacher

*Вес: 1 балл.*

**Постановка.** Сравнение sampler допустимо лишь при одинаковой обученной модели, классах и исходном шуме. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите checkpoint lab 17, запретите случайную замену и проверьте совместимость шагов/T.

**Результат.** Проверяемое состояние «Проверка обученного teacher» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите checkpoint lab 17, запретите случайную замену и проверьте совместимость шагов/T» и его результат доступен следующему этапу.

In [2]:
import math
import torch.nn as nn
import torch.nn.functional as F

# архитектуры и расписание — те же определения, что в работе 17 (веса берутся только из её чекпоинта)
NULL_CLASS = 10
def diffusion_schedule(T, kind="linear"):
    betas = torch.linspace(1e-4, 0.02, T, dtype=torch.float64)
    alpha_bar = torch.cumprod(1 - betas, 0)
    return {"betas": betas.float(), "alphas": (1 - betas).float(), "alpha_bar": alpha_bar.float(), "kind": kind, "T": T}

def q_sample(x0, t, noise, s):
    ab = s["alpha_bar"].to(x0.device)[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

class TimeClassEmbedding(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.cls = nn.Embedding(11, dim)
    def forward(self, t, y):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        ang = t.float()[:, None] * freqs[None]
        return self.mlp(torch.cat([ang.sin(), ang.cos()], 1)) + self.cls(y)

class ResBlock(nn.Module):
    def __init__(self, cin, cout, emb_dim=256):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.emb = nn.Linear(emb_dim, cout)
        self.n2, self.c2 = nn.GroupNorm(8, cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.emb(e)[:, :, None, None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)

class TinyDenoiser(nn.Module):
    def __init__(self, ch=(64, 128, 128)):
        super().__init__()
        self.emb = TimeClassEmbedding()
        self.inp = nn.Conv2d(1, ch[0], 3, padding=1)
        self.d1, self.d2, self.d3 = ResBlock(ch[0], ch[0]), ResBlock(ch[0], ch[1]), ResBlock(ch[1], ch[2])
        self.down1, self.down2 = nn.Conv2d(ch[0], ch[0], 4, 2, 1), nn.Conv2d(ch[1], ch[1], 4, 2, 1)
        self.mid = ResBlock(ch[2], ch[2])
        self.up2, self.up1 = nn.ConvTranspose2d(ch[2], ch[1], 4, 2, 1), nn.ConvTranspose2d(ch[1], ch[0], 4, 2, 1)
        self.u2, self.u1 = ResBlock(2 * ch[1], ch[1]), ResBlock(2 * ch[0], ch[0])
        self.out = nn.Sequential(nn.GroupNorm(8, ch[0]), nn.SiLU(), nn.Conv2d(ch[0], 1, 3, padding=1))
    def forward(self, x, t, y):
        e = self.emb(t, y)
        h1 = self.d1(self.inp(x), e)
        h2 = self.d2(self.down1(h1), e)
        h3 = self.mid(self.d3(self.down2(h2), e), e)
        u2 = self.u2(torch.cat([self.up2(h3), h2], 1), e)
        u1 = self.u1(torch.cat([self.up1(u2), h1], 1), e)
        return self.out(u1)

class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(),
                               nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 10))
    def forward(self, x):
        return self.f(x)

LAB17 = Path("../lab_17_ddpm_cfg/artifacts")
ckpt_file = LAB17 / "ddpm.pt"
if FAST_DEV_RUN and (LAB17 / "ddpm_smoke.pt").exists():   # для smoke годится любой чекпоинт работы 17
    ckpt_file = LAB17 / "ddpm_smoke.pt"
for f in (ckpt_file, LAB17 / "classifier.pt", LAB17 / "classifier_validation.json"):
    if not f.exists():
        raise FileNotFoundError(f"нет {f}: сначала выполните работу 17 — подмена случайными весами запрещена")
ck = torch.load(ckpt_file, map_location="cpu", weights_only=False)
MIN_STEPS = 1 if FAST_DEV_RUN else 10000
assert ck["state"]["step"] >= MIN_STEPS, f"teacher обучен {ck['state']['step']} шагов < {MIN_STEPS}"
assert ck["T"] == 1000 and ck["schedule"] == "linear" and ck["null_class"] == NULL_CLASS
T = ck["T"]
sched = diffusion_schedule(T)
teacher = TinyDenoiser().to(DEVICE)
teacher.load_state_dict(ck["ema"]); teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
clf = DigitClassifier().to(DEVICE)
clf.load_state_dict(torch.load(LAB17 / "classifier.pt", map_location=DEVICE)["state_dict"]); clf.eval()
clf_val = json.loads((LAB17 / "classifier_validation.json").read_text())
teacher_info = {"checkpoint": str(ckpt_file), "train_steps": ck["state"]["step"], "T": T, "schedule": ck["schedule"],
                "weights": "EMA", "classifier_test_accuracy_lab17": clf_val["test_accuracy"]}
save_json("teacher_check.json", teacher_info)
print(json.dumps(teacher_info, indent=1))

{
 "checkpoint": "../lab_17_ddpm_cfg/artifacts/ddpm.pt",
 "train_steps": 10000,
 "T": 1000,
 "schedule": "linear",
 "weights": "EMA",
 "classifier_test_accuracy_lab17": 0.9750000238418579
}


### Воспроизводимый протокол сравнения

*Вес: 1 балл.*

**Постановка.** Сравнение sampler допустимо лишь при одинаковой обученной модели, классах и исходном шуме. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Создайте общие классы и шум, сохраните seed и качество классификатора.

**Результат.** Проверяемое состояние «Воспроизводимый протокол сравнения» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Создайте общие классы и шум, сохраните seed и качество классификатора» и его результат доступен следующему этапу.

In [3]:
import torchvision
mnist_test = torchvision.datasets.MNIST(DATA_ROOT, train=False, download=True)
X_test = mnist_test.data[:2000].float().div(127.5).sub(1).unsqueeze(1); y_test = mnist_test.targets[:2000]
with torch.no_grad():
    clf_acc = float((torch.cat([clf(X_test[i:i + 500].to(DEVICE)).argmax(1).cpu() for i in range(0, 2000, 500)]) == y_test).float().mean())
assert abs(clf_acc - clf_val["test_accuracy"]) < 1e-3, "классификатор из работы 17 должен давать ту же точность"
GUIDANCE = 3.0
PER_CLASS = 2 if FAST_DEV_RUN else 20
y_gen = torch.arange(10).repeat_interleave(PER_CLASS)
NOISE_SEED = SEED + 17
x_T = torch.randn(len(y_gen), 1, 28, 28, generator=torch.Generator().manual_seed(NOISE_SEED))
protocol = {"guidance": GUIDANCE, "per_class": PER_CLASS, "noise_seed": NOISE_SEED, "n_images": len(y_gen),
            "classifier_test_accuracy": clf_acc, "classifier_source": "lab 17 classifier.pt"}
save_json("protocol.json", protocol)
print(protocol)

{'guidance': 3.0, 'per_class': 20, 'noise_seed': 59, 'n_images': 200, 'classifier_test_accuracy': 0.9750000238418579, 'classifier_source': 'lab 17 classifier.pt'}


### Сетка DDIM и граничный переход

*Вес: 1 балл.*

**Постановка.** DDIM сокращает траекторию без переобучения; дополнительный численный solver должен использовать тот же denoiser и CFG. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Реализуйте строго убывающую сетку и DDIM с корректной границей alpha-bar(-1)=1.

**Результат.** Проверяемое состояние «Сетка DDIM и граничный переход» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Реализуйте строго убывающую сетку и DDIM с корректной границей alpha-bar(-1)=1» и его результат доступен следующему этапу.

In [4]:
@torch.no_grad()
def guided_eps(net, x, t, y, w=GUIDANCE):
    """ε_u + w(ε_c − ε_u); NFE считается отдельно для условной и безусловной ветви."""
    eps = net(torch.cat([x, x]), torch.cat([t, t]), torch.cat([y, torch.full_like(y, NULL_CLASS)]))
    e_c, e_u = eps.chunk(2)
    return e_u + w * (e_c - e_u), {"cond": 1, "uncond": 1}

def ddim_grid(steps, T=1000):
    """Строго убывающая сетка шагов T−1 … 0; после последнего шага переход в «−1» с ᾱ=1."""
    grid = torch.linspace(T - 1, 0, steps).round().long()
    grid = torch.unique_consecutive(grid)
    assert (grid[1:] < grid[:-1]).all() and grid[0] == T - 1 and grid[-1] == 0
    return grid

def ab_at(t):
    return torch.tensor(1.0) if t < 0 else sched["alpha_bar"][t]

@torch.no_grad()
def ddim_sample(net, x_T, y, steps, eps_fn=guided_eps, clip_x0=True):
    grid = ddim_grid(steps)
    x, y = x_T.to(DEVICE), y.to(DEVICE)
    nfe = {"cond": 0, "uncond": 0}
    for k, t in enumerate(grid.tolist()):
        s = grid[k + 1].item() if k + 1 < len(grid) else -1
        tt = torch.full((len(x),), t, device=DEVICE, dtype=torch.long)
        eps, n = eps_fn(net, x, tt, y)
        nfe = {key: nfe[key] + n[key] for key in nfe}
        a_t, a_s = ab_at(t).to(DEVICE), ab_at(s).to(DEVICE)
        x0 = (x - (1 - a_t).sqrt() * eps) / a_t.sqrt()
        if clip_x0:
            x0 = x0.clamp(-1, 1)
            eps = (x - a_t.sqrt() * x0) / (1 - a_t).sqrt()   # ε согласуется с обрезанным x̂0, иначе траектория расходится
        x = a_s.sqrt() * x0 + (1 - a_s).sqrt() * eps      # при s = −1: ᾱ=1 и x = x̂0
    return x.clamp(-1, 1).cpu(), nfe

g5 = ddim_grid(5)
print("сетка DDIM на 5 шагов:", g5.tolist(), "→ −1")

сетка DDIM на 5 шагов: [999, 749, 500, 250, 0] → −1


### DPM-Solver++ на той же модели

*Вес: 1 балл.*

**Постановка.** DDIM сокращает траекторию без переобучения; дополнительный численный solver должен использовать тот же denoiser и CFG. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Добавьте solver с тем же epsilon-predictor и beta schedule.

**Результат.** Проверяемое состояние «DPM-Solver++ на той же модели» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Добавьте solver с тем же epsilon-predictor и beta schedule» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Инварианты детерминизма sampler

*Вес: 1 балл.*

**Постановка.** DDIM сокращает траекторию без переобучения; дополнительный численный solver должен использовать тот же denoiser и CFG. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Проверьте конец сетки и совпадение двух запусков DDIM при одинаковом шуме.

**Результат.** Проверяемое состояние «Инварианты детерминизма sampler» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Проверьте конец сетки и совпадение двух запусков DDIM при одинаковом шуме» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Измеримые бюджеты sampling

*Вес: 1 балл.*

**Постановка.** Время и качество измеряются на одном batch; количество NFE учитывает conditional и unconditional обращения отдельно. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Измерьте class agreement и внутриклассовое разнообразие PNG при равном начальном шуме, учитывайте NFE.

**Результат.** Проверяемое состояние «Измеримые бюджеты sampling» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Измерьте class agreement и внутриклассовое разнообразие PNG при равном начальном шуме, учитывайте NFE» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Pareto скорости и качества

*Вес: 1 балл.*

**Постановка.** Время и качество измеряются на одном batch; количество NFE учитывает conditional и unconditional обращения отдельно. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Определите недоминируемые точки только из измеренных времени и точности.

**Результат.** Проверяемое состояние «Pareto скорости и качества» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Определите недоминируемые точки только из измеренных времени и точности» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Подготовка отдельного student

*Вес: 1 балл.*

**Постановка.** Student обучается на последовательных переходах teacher, а не на произвольной регрессии к фиксированному изображению. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите train MNIST и инициализируйте отдельную модель весами teacher.

**Результат.** Проверяемое состояние «Подготовка отдельного student» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите train MNIST и инициализируйте отдельную модель весами teacher» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Двухшаговые цели для distillation

*Вес: 1 балл.*

**Постановка.** Student обучается на последовательных переходах teacher, а не на произвольной регрессии к фиксированному изображению. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Оптимизируйте student по двум переходам замороженного teacher с checkpoint.

**Результат.** Проверяемое состояние «Двухшаговые цели для distillation» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Оптимизируйте student по двум переходам замороженного teacher с checkpoint» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Сравнение при равных и разных NFE

*Вес: 1 балл.*

**Постановка.** Student обучается на последовательных переходах teacher, а не на произвольной регрессии к фиксированному изображению. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сравните teacher 20/10 и student 10 на общих noise/labels; сохраните изображения и время.

**Результат.** Проверяемое состояние «Сравнение при равных и разных NFE» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сравните teacher 20/10 и student 10 на общих noise/labels; сохраните изображения и время» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Творческое мини-исследование

*Вес: 1 балл, бонус сверх десяти обязательных заданий.*

**Постановка.** Контролируемая абляция позволяет отделить влияние одного фактора от шума выборки. Гипотеза формулируется до просмотра отложенных результатов.

Выполните следующие действия:

1. Сформулируйте собственную предметную гипотезу о поведении модели и измените один фактор, сохраняя те же реальные входы, seed и вычислительный бюджет.
2. Сравните с основным экспериментом, не используя test для выбора настройки; объясните наблюдаемые расхождения.

**Результат.** Отдельные код, измеренная таблица и интерпретация в отчёте.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена на реальных входах, сохранены фактические измерения и показана контролируемая разница. Бонус не является обязательным для Run All.

## Анализ результатов

Сопоставьте конфигурации при одинаковых данных, seeds и бюджете. Приведите таблицу фактических метрик и времени, не менее пяти неудачных примеров с объяснением причин, ограничения и проверяемые выводы. Не подменяйте измерения утверждением об успешном запуске.

## Требования к сдаче

Перед сдачей перезапустите ядро и выполните все ячейки по порядку в полном режиме. Приложите конфигурацию, версии библиотек, источники/лицензии данных и модели, checkpoints, таблицу измерений, изображения и краткий анализ ошибок. Для длительного обучения перенесите каталог `artifacts/` на Google Drive и возобновляйте обучение с последнего checkpoint; отсутствие GPU или данных нельзя заменять синтетическими метриками.